In [ ]:
# GPU Check - REQUIRED for BioEmu
import subprocess
import sys

def check_gpu_availability() -> bool:
    """Check if GPU is available (required for BioEmu)."""
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
        if result.returncode == 0:
            print("✅ GPU detected!")
            print(result.stdout)
            return True
        else:
            print("❌ No GPU detected.")
            return False
    except FileNotFoundError:
        print("❌ nvidia-smi not found. Make sure you're in a GPU-enabled environment.")
        return False

# Check GPU
gpu_available = check_gpu_availability()

if not gpu_available:
    print("\n⚠️  WARNING: BioEmu requires GPU for efficient execution.")
    print("   In Google Colab: Runtime > Change runtime type > GPU")

✅ GPU detected!
Tue Feb 17 17:53:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P0             29W /   70W |    2957MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------------------

In [ ]:
import os
import sys
import subprocess
import pathlib

# 1. Variáveis de ambiente que o BioEmu/ColabFold procuram
os.environ["BIOEMU_DISABLE_VENV"] = "1"
os.environ["BIOEMU_FORCE_COLABFOLD_REINSTALL"] = "1"
# Força o ColabFold a não procurar o diretório padrão do Google Drive do Colab
os.environ["COLABFOLD_RESOURCES"] = "/kaggle/working/colabfold_resources" 

# 2. Instalação Cirúrgica (Kaggle já tem mamba, não precisa de condacolab)
print("🛠️ Instalando BioEmu e dependências...")
!pip install bioemu==0.1.6 MDAnalysis colabfold --quiet

# 3. O Patch do Arantes (Versão Kaggle/Python 3.12)
# Em vez de editar o arquivo físico (que mata o kernel), alteramos a função na RAM.
import pathlib
import os

original_fspath = os.fspath
def patched_fspath(path):
    # Resolve o erro onde o BioEmu passa um objeto Path em vez de String
    if isinstance(path, pathlib.Path):
        return str(path)
    return original_fspath(path)

os.fspath = patched_fspath
print("✅ Patch de compatibilidade aplicado em memória.")

# 4. Dependências de Sistema (AmberTools é obrigatório para o BioEmu)
print("🛠️ Instalando AmberTools via Mamba...")
!mamba install -c conda-forge ambertools -y --quiet

# 5. Importação e Verificação
try:
    import bioemu
    import MDAnalysis
    print("🚀 BioEmu carregado com sucesso!")
except ImportError as e:
    print(f"❌ Erro ao importar: {e}")

✅ BioEmu instalado! Agora reinicie o kernel e importe.


In [ ]:
#@title ### **Import Google Drive**
#@markdown Click in the "Run" buttom to make your Google Drive accessible.
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
Google_Drive_Path = '/content/drive/MyDrive/MutationConformationsBioEmu' #@param {type:"string"}
workDir = Google_Drive_Path


Mounted at /content/drive


In [ ]:
from bioemu.sample import main as sample

# --- Parâmetros ---
#@markdown - `sequencia_nativa`: Monomer sequence to sample
sequencia_nativa = "MSYPGYPPPPGGYPPAAPGGGPWGGAAYPPPPSMPPIGLDNVATYAGQFNQDYLSGMAANMSGTFGGANMPNLYPGAPGAGYPPVPPGGFGQPPSAQQPVPPYGMYPPPGGNPPSRMPSYPPYPGAPVPGQPMPPPGQQPPGAYPGQPPVTYPGQPPVPLPGQQQPVPSYPGYPGSGTVTPAVPPTQFGSRGTITDAPGFDPLRDAEVLRKAMKGFGTDEQAIIDCLGSRSNKQRQQILLSFKTAYGKDLIKDLKSELSGNFEKTILALMKTPVLFDIYEIKEAIKGVGTDEACLIEILASRSNEHIRELNRAYKAEFKKTLEEAIRSDTSGHFQRLLISLSQGNRDESTNVDMSLAQRDAQELYAAGENRLGTDESKFNAVLCSRSRAHLVAVFNEYQRMTGRDIEKSICREMSGDLEEGMLAVVKCLKNTPAFFAERLNKAMRGAGTKDRTLIRIMVSRSETDLLDIRSEYKRMYGKSLYHDISGDTSGDYRKILLKICGGND" #@param {type:"string"}

#@markdown - `mutacao_input`: Mutation to create
mutacao_input = "P36R" #@param {type:"string", min:3, max: 4}

#@markdown - `num_samples`: Number of samples requested
num_samples = 200 #@param {type:"slider", min:10, max:5000, step:10}
model_name = "bioemu-v1.1"

# --- Execução Nativa ---
print(f"Gerando Nativa (Salvando em {workDir}/out_native)...")
output_dir_native = f'{workDir}/out_native'

smp = sample(
    sequence=sequencia_nativa,
    num_samples=num_samples,
    model_name=model_name,
    output_dir=output_dir_native,
    filter_samples=True
)

# --- Lógica da Mutação ---
aa_original = mutacao_input[0]
posicao = int(mutacao_input[1:-1]) - 1 # Correto: converte para índice 0-based
aa_mutado = mutacao_input[-1]

# Verificação de segurança
if sequencia_nativa[posicao] == aa_original:
    sequencia_mutada = sequencia_nativa[:posicao] + aa_mutado + sequencia_nativa[posicao+1:]
    print(f"\n---")
    print(f"Mutação {mutacao_input} verificada e aplicada.")
    print(f"Resíduo {sequencia_nativa[posicao]} na posição {posicao+1} substituído por {aa_mutado}.")
    print(f"---")

    print(f"Gerando Mutante {mutacao_input}...")
    output_dir_mutant = f'{workDir}/out_mutant_{mutacao_input}'
    smp = sample(
        sequence=sequencia_mutada,
        num_samples=num_samples,
        model_name=model_name,
        output_dir=output_dir_mutant,
        filter_samples=True,

    )
else:
    # Isso evita que você rode horas de simulação na proteína errada
    raise ValueError(f"ERRO CRÍTICO: O resíduo na posição {posicao+1} é '{sequencia_nativa[posicao]}', mas você pediu para mudar '{aa_original}'. Verifique a numeração da sequência FASTA.")

Gerando Nativa (Salvando em /content/drive/MyDrive/MutationConformationsBioEmu/out_native)...


Sequence MSYPGYPPPPGGYPPAAPGGGPWGGAAYPPPPSMPPIGLDNVATYAGQFNQDYLSGMAANMSGTFGGANMPNLYPGAPGAGYPPVPPGGFGQPPSAQQPVPPYGMYPPPGGNPPSRMPSYPPYPGAPVPGQPMPPPGQQPPGAYPGQPPVTYPGQPPVPLPGQQQPVPSYPGYPGSGTVTPAVPPTQFGSRGTITDAPGFDPLRDAEVLRKAMKGFGTDEQAIIDCLGSRSNKQRQQILLSFKTAYGKDLIKDLKSELSGNFEKTILALMKTPVLFDIYEIKEAIKGVGTDEACLIEILASRSNEHIRELNRAYKAEFKKTLEEAIRSDTSGHFQRLLISLSQGNRDESTNVDMSLAQRDAQELYAAGENRLGTDESKFNAVLCSRSRAHLVAVFNEYQRMTGRDIEKSICREMSGDLEEGMLAVVKCLKNTPAFFAERLNKAMRGAGTKDRTLIRIMVSRSETDLLDIRSEYKRMYGKSLYHDISGDTSGDYRKILLKICGGND may be too long. Attempting with batch_size = 1.
